# Домашняя работа 6: Задачи на графах

##  ФИО Студента: Фоменко Лиза

В этом домашнем задании вы будете работать с набором данных MUTAG, который испольщуется в задачах классификации графов. Он состоит из графов, представляющих химические соединения, целью которых является классификация соединений на основе их мутагенного эффекта на бактерии `Salmonella typhimurium`.

---

Описание набора данных

Вершины: Каждая вершина представляет атом в молекуле (например, углерод, кислород, азот).

Ребра: Каждое ребро представляет химическую связь между атомами.

Метки: Каждый граф помечен как мутагенный (1) или немутагенный (0), задача логистической регрессии.

Импортируем необходимые модули

In [1]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.6 MB/s eta 0:00:00


In [2]:
import torch.nn.functional as F
import numpy               as np
import networkx  as nx

import plotly.graph_objects as go

import torch

from torch_geometric.datasets import TUDataset
from torch_geometric.loader   import DataLoader
from torch_geometric.nn       import GCNConv, GINConv, global_mean_pool
from torch_geometric.data     import Data

import torch_geometric.transforms as T

from torch.nn import Sequential, Linear, ReLU

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(42)

Функция визуализации графа

In [3]:
def visualize_graph(data, graph_id):
    G = nx.Graph()
    edge_index = data.edge_index.numpy()
    G.add_edges_from(edge_index.T)

    node_labels = data.x.argmax(dim=1).numpy() if data.x is not None else None
    pos = nx.spring_layout(G, seed=42)

    edge_x = []
    edge_y = []
    for edge in G.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=1, color='black'),
        hoverinfo='none',
        mode='lines'
    )

    node_x = [pos[node][0] for node in G.nodes()]
    node_y = [pos[node][1] for node in G.nodes()]

    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        marker=dict(
            size=10,
            color=node_labels if node_labels is not None else 'blue',
            colorscale='Viridis',
            showscale=True
        ),
        text=[str(node) for node in G.nodes()],
        textposition='top center',
        hoverinfo='text'
    )

    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(
        title=f"Graph {graph_id} - Label: {data.y.item()}",
        showlegend=False, height=600, width=600,
        margin=dict(b=0, l=0, r=0, t=40),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
    )

    fig.show()

Загрузим набор данных

In [4]:
dataset = TUDataset(root='data/TUDataset', name='MUTAG')

Processing...
Done!


#### Задача 1 [1 балл]

Визулизируйте случайные примеры из набора данных `MUTAG`. Разбейте выборку на тренировочную и тестовую.

In [5]:
for idx in range(5):
    visualize_graph(dataset[idx], idx)

In [ ]:
# я в виду проф деформации и невероятной любви к химии не могу оставить молекулы такими... поэтому я порыскала в документации MUTAG - разобралась в кодировке атомов, обозначении связей
# (признаюсь, с участием gpt) написала функцию для отрисовки молекул такими, какими мы привыкли их видеть

In [6]:
def visualize_molecule(data, graph_id):
    G = nx.Graph()

    edge_index = data.edge_index.cpu().numpy()
    edge_attr  = data.edge_attr.cpu().numpy()

    G.add_edges_from(edge_index.T)


    # вот эта схема, в принципе, отражает привычные цвета для атомов https://sciencenotes.org/molecule-atom-colors-cpk-colors/
    atom_types = ['C', 'N', 'O', 'F', 'I', 'Cl', 'Br']
    atom_colors = {
        'C': 'gray',
        'N': 'blue',
        'O': 'red',
        'F': 'green',
        'Cl': 'green',
        'Br': 'darkred',
        'I': 'purple'
    }

    atom_idx    = data.x.argmax(dim=1).cpu().numpy()
    atom_labels = [atom_types[i] for i in atom_idx]
    node_colors = [atom_colors[a] for a in atom_labels]

    bond_types = edge_attr.argmax(axis=1)

    pos = nx.spring_layout(G, seed=42, k=0.35)

    edge_traces = []

    def draw_bond(xa, ya, xb, yb, dash='solid', width=2):
        edge_traces.append(
            go.Scatter(
                x=[xa, xb],
                y=[ya, yb],
                mode='lines',
                line=dict(color='black', width=width, dash=dash),
                hoverinfo='none'
            )
        )

    def bond_offsets(x0, y0, x1, y1, spacing=0.06):
        dx = x1 - x0
        dy = y1 - y0
        length = np.hypot(dx, dy)
        if length == 0:
            return (0, 0), (0, 0)

        nx = -dy / length * 0.5
        ny = dx / length * 0.5

        return (nx * spacing, ny * spacing), (-nx * spacing, -ny * spacing)


    for (u, v), bond in zip(edge_index.T, bond_types):

        if u > v:
            continue

        x0, y0 = pos[u]
        x1, y1 = pos[v]

        (ox1, oy1), (ox2, oy2) = bond_offsets(x0, y0, x1, y1)

        if bond == 1:  # single -> one line
            draw_bond(x0, y0, x1, y1)

        elif bond == 2:  # double -> draw 2 lines
            draw_bond(x0 + ox1 * 0.5, y0 + oy1 * 0.5, x1 + ox1 * 0.5, y1 + oy1 * 0.5)
            draw_bond(x0 + ox2 * 0.5, y0 + oy2 * 0.5, x1 + ox2 * 0.5, y1 + oy2 * 0.5)

        elif bond == 3:  # triple -> draw 3 lines
            draw_bond(x0, y0, x1, y1)
            draw_bond(x0 + ox1, y0 + oy1, x1 + ox1, y1 + oy1)
            draw_bond(x0 + ox2, y0 + oy2, x1 + ox2, y1 + oy2)

        elif bond == 0:  # aromatic -> solid line + dashed line
            draw_bond(x0, y0, x1, y1)
            draw_bond(x0 + ox1, y0 + oy1, x1 + ox1, y1 + oy1, dash='dash')


    node_x = [pos[i][0] for i in G.nodes()]
    node_y = [pos[i][1] for i in G.nodes()]

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode='markers+text',
        text=atom_labels,
        textposition='middle center',
        marker=dict(
            size=22,
            color=node_colors,
            line=dict(color='white', width=1)
        ),
        hoverinfo='text'
    )

    fig = go.Figure(edge_traces + [node_trace])
    fig.update_layout(
        title=f"Molecule {graph_id} — Mutagenic: {data.y.item()}",
        showlegend=False,
        height=600,
        width=600,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        margin=dict(l=10, r=10, t=40, b=10)
    )

    fig.show()


In [7]:
for idx in range(5):
    visualize_molecule(dataset[idx], idx)

In [ ]:
# ну вот теперь красота!!!!! я правда хз, как может фтор быть в ароматическом цикле? не очень поняла составление MUTAG, но все-таки напрямую к дз это не относится

In [8]:
# делим выборку

from torch.utils.data import random_split
train_dataset, val_dataset, test_dataset = random_split(dataset, [0.8, 0, 0.2])


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataset.py:469: UserWarning:

Length of split at index 1 is 0. This might result in an empty dataset.



In [9]:
len(train_dataset), len(test_dataset)

(151, 37)

In [10]:
train_loader = DataLoader(train_dataset, shuffle = True, batch_size = 16, drop_last = True)
test_loader = DataLoader(test_dataset, shuffle = False, batch_size = 16, drop_last = False)

#### Задача 2 [2 балла]

Реализуйте модель `GCNConv` согласно предлагаемой архитектуры, используйте пакет `torch-geometric`

<p style="text-align: center"><img src="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAcMAAAEsCAIAAAAw9k/eAAAoeUlEQVR4Xu2dCXgUVbqG/0GuGFD0
EmUHb6Iim8hVgjKiwLijIqsLiGQUgoIiLkFGREAEL+GOShDQyEAeF2RxAUTgyiYoRGNGXAYddRAF
QRTcBjGy9v1OTlndOVXdpCgip8n3Pt/Tz6m//jq15PRX/6lqVCKEEELCIWaAEEJIQOikhBASFjop
IYSEhU5KCCFhoZMSQkhY6KSEEBIWOikhhISFTkoIIWGhkxJCSFjopIQQEhY6KSGEhIVOSgghYaGT
EkJIWOikhBASlmBOKoQQUmEwHTA+AVIjJU5asglFUdQRLjopRVFUWNFJKYqiwopOSlEUFVZ0Uoqi
qLCik1IURYUVnZSiKCqs6KQURVFhRSelHP36q2zfbgYt1PffK3njibV3r2zaJD/9ZMaTWps3mxFX
27bJN9+YQar8RCctX61ZI926ySmnSOPGcvXVMmeO7NvnrHrjDencWdLSpHlzycyU99934mhfdZXs
2uUsLlsmWVnRDuNtFV6DB8uUKWbQQt19t9x7rxlMrDfflLp1JSNDnnvOXJVY06fLt9+awfLWokVy
wQXSsKG0bStr15ZahTvB5ZdLdrazeMklsnKlubnWww/LsGFmMFIyutq1k/btpXt3mTxZ9uwxE4zk
114zg5ddJh98EF2cOVPuusvMqYCik5ajXnxRjj1Wxo2Tdevk669l/nw57zzV0KuqVZPx4+Wf/5SN
G+Wpp6R3b2erZs3k6KNlwgRn8fnn5eyzox3G2yqktmyRevVk924zbqEOwknhCLho3vgBhWvyzjtm
sFwFZ0xNlWeflfXrZfly+fTTUmtxT23aVC691FlcskTOP9/sQSuek2J0jRwpBQUya5bUry+jRpkJ
RvIzz5hBjEDczt1FXNgLLzRzKqDopOUluFLt2lFD1EJBimkmVtWpI488UmqVWx1g+N5+u9SsKTt2
qEXXSRNshQIWA7pHD7njDtmwwVk1fLi8954MGqSqD/f7ADfHV8jdfPFiZ9XYsTJgQDTuuy1Kj3//
22m/9JK8+mo0GaUTjrlrV9U/Dmb0aNXGue/fH+3TkO8ufE8E2rlT5XfpIo8+Knfe6TgpOp86VXr2
lH79pLDQ7N/VE09IerpcdJHqE7cx363mzZObb1bF/pAh0UkxClLcCK+/Xm0I70hw+u++q06kVy+1
6Ns/PPHGG9WkBFfp88/NI4wVnHHiRDPodtKhg/pLuU6K4YQx9tFHZmYkoZO6V3vMGGnTxmn7Hjad
tOyik5aX3npLXVzfJ3pvv61WffedGdfC8J0zR03idL3gOmmCrTp1kiuvlBUr1NcMFc3WrSpYq5a0
aqUmX+gNwaVLVfD116VBg+gTBiTMmKEamE7qhpbvtkcd5RTUEMzF/aIiuXVrVS/n50tKivK7xx+X
hQvVI4jYPg357sL3RCBcDdgr6rX775caNRwnxXe+Y0eVjE5wjzEmwq6KipRfDB6sZs24OflulZMj
r7yiHsXASZs3V3c7BLHqxBMlN1dt+OWXiU6/ZUtVReKUfY8Kt4STTpIXXlCV4LRpcY8zUnKzxF6w
ob4OsHJ3Fe4lGBgoUWGRrpNCuA/h7uLtqixOilIdO9Jt72Ebya7opL6ik5aX5s5VY85dxHd14ECl
9983VxnSToqhfPzx6r2B66Txtvr4Y6laVX780VlEYaUtGN9wfHt1EN/8oUNVA6UHCjTMCtFet07t
orhYtVEC43vu9um7bQIrgY3q9sUXq7JLt1Hy9O0b7dOQdxfxTgQ1F2rDn3924uedp5wUlR2uBvxF
B3F54QXevWihGHzqKdVIsBVcbONG+ewzNaPHldHB2Nl9gtOHjeq2b/+rVsmpp8oPPzjBBILngjPO
UPcV+DL2joJar8KdAHeXSIlFxjppdrbccovZj06L56QZGerZPT6bNnXqWd/D1sl00jKKTlpeWrZM
/vAH+eUXZxFeM2WKqk1efllN04C7ypB2UjSuu07NZF0njbcVJtTYxF188EFnmolvuPtmANNt9/s2
cqTccINq3HOP9O/vBKtXV1NUtxPfbRNYiZt87bWqiNNtnC8qJrdPQ95dxDsRTL3PPDMaHzBAOemC
BVKlisqHHTRpogptlFTevWi5ThpvK5S69eurufMVV6hLgb+d3rCMTuqWmb79Ywbw5z+rmwFm7o89
Fn2X6NWmTepP7L4WQzJuG2hguo0+4cWoqXF3ueii6H1l+HDp08fsJ5LQSe++W13qESPUkwHcOSJx
Dlsne530uONKveYaN07dPr07qmiik5aXfvpJjjkm+jRN65RTlJPqVTAI71aRGCfFKNcvrLSTxtsK
JU/dutFFFC+33aYa+IZ/+KEThFW5pqkLEHwtMYlz69BGjUq9pfXdFjN3VG06eNNNpazETYaTuo/5
4KSofdw+DXl3Ee9E8L09+eRovGdP5aSrVyub83brK9dJfbdCBeo+lYZwDLpmh2CvrpOW5fR9+9dC
/xgMuCUkeMkDz8VffPFiZxEu1qKFakybpmYPWhgDlSurneoc3IF8378lcFLXHFHPahOMd9i+Tnr6
6aUeteOmkplp5lRA0UnLURjKsE63YMF3CeMVTqpXpaVFV8HdMKXSbddJIfgLylj33b3vVihP3Pn1
1q2q0NAP7LxWpdtQu3bK4xo3jkbwZYBlu4u+22LWqaexW7Yo6/G1kjBOGu9EEMdF0P4CL0PNCO8o
LlaXIi/P6QGLCd7kuE7quxUqPuxXP+WYPVt9JVwnhfHpv1ekbKfv2z/y3UcWsfMAnN3cuU7bFf4Q
vXurhzB796oCeeBAM8GY3bdu7dOJTjugk373nVNg+h62kewK1onpvJ4bffWVuh9j2mTkVEDRSctR
+D489JB6Q4IyB/Mm2AFmpvr3iag+MHs94QQ1k0I9iNonP9/ZKtZJ8SVEKeQ6abytMBtFu2VL9ZbG
9xtuOOn06eoPjy+bG4FPtW0bXfTdFuUwaiLsBccDi/TdURgnjcQ5EQjV3Iknqkd7qNFgi7oKwx0F
i7gOrVqpKxz7fsaQ66S+W+HP1LWr2i/Oq0sX9ZdynRQegUsNT0dVWJbT9+0f1xZjAFtB+OPqCXWk
xDS9Rrltm5x7rvK1hg2VYXnfWMY66ebNalDpe4A37YBOCg0dqn5bGvE7bJ0ci36MgLsCLhcGYZMm
yoiHDDF3UTFFJy13objAHR53b+9PgvbsUT8bhF16t0og363Q+Rdf+H+pyiJsDo+IdQRfoRL517+i
r/7LQ/FOZPdutWvvL8lxZ9qwIfAvYb1b4Q+U4J8MaZX99I3+d+1Sf7JNm0qNATjRJ5+YG2p9+aWy
VG/c0KhR6jmpNx6J76SJ5b0s8bRzpzqjsmRWENFJKUcFBdFamCpv4f7qfeQdVCNGRB/vGjo4J6UO
WnRSqnyFQg+TREMHrP4OTr/nvizX1KnRH1FQv4PopBRFUWFFJ6UoigorOilFUVRY0UkpiqLCik5K
URQVVnRSiqKosKKTUhRFhRWdlKIoKqzK10kJIaSCYDpgfAKkEnJYCDSgCTkscIwS26GTEvvhGCW2
Qycl9sMxSmyHTkrsh2OU2A6dlNgPxyixHTopsR+OUWI7dFJiPxyjxHbopMR+OEaJ7dBJif1wjBLb
oZMS++EYJbZDJyX2wzFKbIdOSuyHY5TYDp2U2A/HKLEdOimxH45RYjt0UmI/HKPEduikxH44Ront
0EmJ/XCMEusoKCjIy8vbt2+fXnSdFBHEsTaaSogd0EmJdcAx09PTMzIyCgsLI785KdqIIO46LCH2
QCclNpKTkwMDrVSpUlZWFhr4RBsNxM1UQiyATkpsZPv27SkpKVIaRBA3UwmxADopsZTMzEzDSREx
kwixAzopsZTCwkLDSfVjU0IshE5K7CUjI8O1UbTN1YRYA52U2Et+fr7rpGibqwmxBjopsZfi4uLU
1FTYKD7RNlcTYg10UmI12dnZcFJ8misIsQk6KbGa9evXV65cGZ/mCkJsgk5KbGfSpElmiBDLCOak
7uN/Qgg54jEdMD4BUiPOv4CmKIo68kUnpSiKCis6KUVRVFjRSSmKosKKTkpRFBVWdFKKoqiwopNS
FEWFFZ2UoigqrOikVBLo119l+3Yz+Ltp82YzorVtm3zzjRmkKqbopBVXa9ZIt25yyinSuLFcfbXM
mSP79jmr3nhDOneWtDRp3lwyM+X991UQjauukl27nJxlyyQrK9qb7yaHSoMHy5Qp0cVFi+SCC6Rh
Q2nbVtaujcZ/+kkuv1yys83NvYrXg28nl1wiK1eaPUAPPyzDhpnBSMmFatdO2reX7t1l8mTZs8dM
8Oa/9lqpyGWXyQcfRBdnzpS77jK3oqwSnbSC6sUX5dhjZdw4WbdOvv5a5s+X885TDb2qWjUZP17+
+U/ZuFGeekp691bxZs3k6KNlwgSnh+efl7PPjvbmu8kh0ZYtUq+e7N7tLMLUUlPl2Wdl/XpZvlw+
/TSaCWdv2lQuvdTswVCCHnw7WbJEzj/f7CQS30lxoUaOlIICmTVL6teXUaPMBG/+M8+UiuBi4s7k
LuLCXnihuRVlleikFVFwpdq1o56ohYJ07161qk4deeSRUqt0VYUv/O23S82asmOHWnSdNMEmEGpY
GEGPHnLHHbJhgxMcPlzee08GDVJVm2sicHNYj9vD4sXOqrFjZcCAaBymNnFidNEVPLFDB5V8QCeN
10MkTie4MrhcH31kJidwUvekxoyRNm2c9v79MnWq9Owp/fpJYaF/vhadNOlEJ62Ieust9Yf//nsz
Dr39tlr13XdmPFLyhZ8zR818dZHlOmmCTaBOneTKK2XFCmVPqAS3blXBWrWkVSs1aUWHCC5dqoKv
vy4NGkSfMCBhxgzVwDRcNyIlrn3UUWpD3e306U585051eKgu4W6JnTReD4k7geM/+qjZVVmcFDN3
7Ei3YaAdO6pLgb3j3uM+VaCTHgGik1ZEzZ2rvqvuYk6ODByo9P775qpYaSfF9//449XLFtdJE2zy
8cdStar8+KOz2Lmz48Jw0hdecIKoVYcOVQ2UbOnpaiqN9rp1ai/FxaqNKhgzZZ2MqhaccYYy34UL
1az/iSdUfPBg5dSREndL7KTxekjcSXa23HKL2VUCJ83IUM+g8dm0qVPMfv65ukowa52Daw5jdfPp
pMkuOmlF1LJl8oc/yC+/OIsvvqje55x0krz8spreAndVrLSTonHddXLnnVEnTbAJJuzYyl188EHp
1Us14KTuG5XRo6MmNXKk3HCDatxzj/Tv7wSrV5d333XamzapfT33nLP42GPq8S5mytjLDz+oxw5w
6osukp9/ju7UkG8PaCTuZPhw6dPH7CqBk959tzr3ESPUY4HPPlPBBQukShW1Ct7apImqvlGfuvmG
kx53XKl3XOPGycUXm3uhrBKdtCLqp5/kmGPk1VdLBU85RTmpXjVvnrlJJMZJYQ36bZV20gSbrFol
detGF1H03XabasBJP/zQCcJJXdPUhRvsDJNftw5t1Cj6ahtzf+x68WJnEQbUooVMm6YKWC0cSeXK
qn93p4Z8e0AjcSfw+nvvNbtK4KSuM6KY1Sa4erWqf73JRr7W6aeXemSMsj0z09yKskp00goqWACs
031Uh0IM33M4qV6VlhZdBXfDVDQS46QQvA81rPvuPt4mKOvgR6h50d66VRVomFBH4jsp1K6dmhc3
bhyNwETg2rGLvXurRwF798oVV6iHEu6qiGdijt3NnVsq4YA9eDuBWrf26acsTvrdd06BWVysLlFe
nhPHIq6SN18L1onpvC7zv/pK3VcwAzD2QlklOmkFFXzkoYekRg1VM2K+CVscMEC+/VatQtWGafgJ
J6gZKOrB+vUlP1/FY510yxZJSYk6abxNIiVPErDYsqV6s+T6TgInnT5dDUqYlBtB/di2bXRx2zY5
91zlSg0bKrsx3psZJgjT9Bpl4h68nWzerK6PfmhrpB3QSaGhQ9VvS9HAnQb1L65Pq1bqsrsvu5Af
S58+6sly167qejZpoox4yBBzF5RtopNWaKEoQ2WEqgfGaqzas0f93BKO6d0qnuJtgs6/+MLHicoo
bA4jdp1X68svlSF6kw3BiT75xAxqlbEHaNQo9ZzUG4/npImF29WGDdGfxybQzp3qepYlkzrsopNS
SaCCgmg5XHbhPuH79DaoRoxwfkJr6OCclDoiRSelqIPU1KmSm2sGqYopOilFUVRY0UkpiqLCik5K
URQVVnRSiqKosKKTUhRFhRWdlKIoKqzopBRFUWFFJ6Uoigqr8nVSQgipIJgOGJ8AqYQcFgINaEIO
CxyjxHbopMR+OEaJ7dBJif1wjBLboZMS++EYJbZDJyX2wzFKbIdOSuyHY5TYDp2U2A/HKLEdOimx
H45RYjt0UmI/HKPEduikxH44Ront0EmJ/XCMEtuhkxL74RgltkMnJfbDMUpsh05K7IdjlNgOnZTY
D8cosR06KbEfjlFiO3RSYj8co8Q6CgoK8vLy9u3bpxddJ0UEcayNphJiB3RSYh1wzPT09IyMjMLC
wshvToo2Ioi7DkuIPdBJiY3k5OTAQCtVqpSVlYUGPtFGA3EzlRALoJMSG9m+fXtKSoqUBhHEzVRC
LIBOSiwlMzPTcFJEzCRC7IBOSiylsLDQcFL92JQQC6GTEnvJyMhwbRRtczUh1kAnJfaSn5/vOina
5mpCrIFOSuyluLg4NTUVNopPtM3VhFgDnZRYTXZ2NpwUn+YKQmyCTkqsZv369ZUrV8anuYIQm6CT
EtuZNGmSGSLEMoI5qfv4nxBCjnhMB4xPgNRIzH9LghBCjmwC2V2A1EjArgkhJHkJZHcBUiMBuyaE
kOQlkN0FSI0E7JoQQpKXQHYXIDUSsGtCCEleAtldgNRIwK4JISR5CWR3AVIjAbsmhJDkJZDdBUiN
BOyaEEKSl0B2FyA1ErBrkuxs2rRp165dZjSGjRs37t6924zG4YC9JSNH5EkRTSC7C5AaCdg1SXZO
Pvnkd955x4zGUKtWrQ8//NCMxuGAvSUjR+RJEU0guwuQGgnYNUl2DmgTdNIj8qSIJpDdBUiNBOya
JAs7duy47777unTpkpub+/LLL7/yyis67toEJrDjx4/v0aPHHXfcsWHDBndDOOmiRYtuvfXW6667
7sUXX9TBefPm3XzzzZ07dx4yZMg333zjJicwneHDh69du/b222/v2rXr/PnzsbvRo0ejPWHChP37
9+scNKZOndqzZ89+/fq5/xsS332ht/fee2/QoEHdu3d/5plndNAXZOKQjOOPxDlf32CCkyLJTiC7
C5AaCdg1SRbat29//fXXr1y58sEHH0xNTb333nt13LWJTp06XXnllStWrBg7diwStm7dqhPgpM2b
N3/ppZfgv/Xq1Zs1a1ak5H+wDC9es2YN3A1r9+7da/TmBf20bt0aXpafn5+SkgJPf/zxxxcuXJiW
ljZjxgydAwPt2LEjjmHmzJl16tSB80bi7Au9tWrVCmlz5szB0S5dujR2X7H4Hn8kzvn6BhOcFEl2
AtldgNRIwK5JUvDBBx9Ur179l19+0YsdOnQwnPTjjz+uWrXqjz/+qIMoAEeNGqXbcKK//e1vuj19
+vRzzjlHt3fv3r1x48bPPvsM9rRu3TodTGA66MctCS+++GIUp7o9ZsyYvn37ovH5559Xq1Zt586d
Og4DhbHqtndf6O2FF17Qa1E/Dh06VLe9+B6/7/n6BiMJT4okO4HsLkBqJGDXJCnABLlly5bu4sCB
Aw0nxXS7WbNmbgLq1l69euk2nMj1kXfffbdGjRpo3H///fXr14cjX3HFFfDoZcuW6YQEpoN+YOi6
fe211+bm5ur2lClTMENHY8GCBVWqVMFhNG3atEmTJg0aNEB9Gomzr9jeRo8efcstt+i2F9/j9z1f
32Ak4UmRZCeQ3QVIjQTsmiQFb775Zt26dd3FHj16GE66atWq2ITBgwffdtttug0nWrJkiW4vX748
PT0dVWHNmjV37Nihg9jQTUhgOrFvruCkEydO1G04abdu3dBYvXo1Sk43XxNvX7G9wUn79+/vbmLg
PX40fM/XNxhJeFIk2QlkdwFSIwG7JknBrl27GjZs+OSTT0ZK/hfzmEQbTvrzzz+7s++tW7fWrl17
4cKFOgFxOO/+Eq655poBAwagBwT1/71u9uzZGDCHxEnRYVpaWl5eno5jEfP9ePsK5KTG8SPoe76+
wUjCkyLJTiC7C5AaCdg1SRb+/ve/n3XWWampqZdeemnPnj0xddVx1yYwa8YkumXLlsgZNmyYuyHM
Be6D6fZpp53Wpk2bbdu2wZK6du2K5LPPPrtLly6YjB8SJwVr165t0aJFo0aNWrVqhfJw+vTp8fYV
yEmN49dx3/P1DSY4KZLsBLK7AKmRgF2TZKR169buK+xYYFtffPGF7/8q+Ycffti0aVNs5Kuvvtq8
eXNs5FDx7bffbtiwIfYfVoXZl/Zc7/FH4pyvb5AcqQSyuwCpkYBdk2QhNzf3vvvumzBhQqdOnVBz
lf0fgCY7sdUrIQaB7C5AaiRg1yRZQJU3efLksWPHzpw589dffzVXHzpQPDbzcNAVZdmJt9/x48fH
/tsBQmIJZHcBUiMBuyaEkOQlkN0FSI0E7JoQQpKXQHYXIDUSsGtCCEleAtldgNRIwK4JISR5CWR3
AVIjAbsmhJDkJZDdBUiNBOyaEEKSl0B2FyA1ErBrQghJXgLZXYDUSMCuCSEkeQlkdwFSIyVdE0JI
BcF0wPgESCXksBBoQBNyWOAYJbZDJyX2wzFKbIdOSuyHY5TYDp2U2A/HKLEdOimxH45RYjt0UmI/
HKPEduikxH44Ront0EmJ/XCMEtuhkxL74RgltkMnJfbDMUpsh05K7IdjlNgOnZTYD8cosR06KbEf
jlFiO3RSYj8co8R26KTEfjhGie3QSYn9cIwS26GTEvvhGCXWUVBQkJeXt2/fPr3oOikiiGNtNJUQ
O6CTEuuAY6anp2dkZBQWFkZ+c1K0EUHcdVhC7IFOSmwkJycHBlqpUqWsrCw08Ik2GoibqYRYAJ2U
2Mj27dtTUlKkNIggbqYSYgF0UmIpmZmZhpMiYiYRYgd0UmIphYWFhpPqx6aEWAidlNhLRkaGa6No
m6sJsQY6KbGX/Px810nRNlcTYg10UmIvxcXFqampsFF8om2uJsQa6KTEarKzs+Gk+DRXEGITdFJi
NevXr69cuTI+zRWE2ASdlNjOpEmTzBAhlhHMSd3H/4QQcsRjOmB8AqRGtJMWRSiKoo580UkpiqLC
ik5KURQVVnRSiqKosKKTUhRFhRWdlKIoKqzopBRFUWFFJ6UoigorOilVJi3cJAW7zOAh1JpfZdl2
M/i7adFmM6K1dJu89o0ZpCiv6KSUj67KlEmvlYrUOVmefsdMO4TqOVj+MiW6mLtIzrpAajeUlm1l
xtpofOVP8sfL5cZsc3Ovbhomjc9SPbRoIw89pyJv7ZZ+D6jFumly7iUy9Y1oMhbzVpo9QLc9rPrx
xnF9zmonZ7eXC7vL0Mny9h4zwZtvXE/oj5fJzA+ii2NnSq+7zBwqWUQnpXyU3kwefKZUZHqBrPq3
mXaotHiL1KynnE4vwtSOT5XRz8q89fLEcnnp02hm1yxJbyptLjV78OrhWcr6X/lCxs2RY6oq38Tx
w9GeXKG6Hfy/klJNFnzpJE9eIv99vtlDUXwnxfXJGqmuCfZSs770H2UmePON6wnhAGLd/I7x0vpC
M4dKFtFJKR95v/kwi1c3Ou2+w2XGe3LdIFWRuWnv7JfhU+WyntKlnzxdGN3wkXly9c3SvrPcOCQ6
U0YPz72reri8l1ocOFZ6DIhuAlMbMjG66Aqu2qqDSi6Lk8YK5a23wwanKh/U7cJ9klpb5nxk5iRw
UvfEB4xRda5ux7sI3utZRCc9skQnpXzk/ebHzu5r1JKmrdRsFOUeisfJS1UQ3nFeR1XxIX5ineiU
fFCOPPqKTFujnPSU5lK41+mhUUtVdeYuVItwujEznHxUppWOUp1c0EnOv1JGTHfib+5UR4X6FO5W
Rid9bavM/oeMzFdl4/wNpVYt3Sb/cbS6H7gR3BXuetTsoSxOijoXh6rb8S6C93oW0UmPLNFJKR95
v/mGk+a84LSvv0Myh8r8z5UvwOx0EO4JT3G3hTminn35MzWFn73O6QE26ibUqKlmyroNywOnnqEM
Gj6LTe57QsV7DlbVaFGJu5XRSZGJE8HU/ub7VbXoxt/eo3q4sk+p5BuzpdstPj3Ec9KmGfKnbuoz
valTzCa4CN7rWUQnPbJEJ6V85P3mG07qviq5dbQyoMcWyNFV1FawlbQmUquBKs10AlwMJSFm5W2v
kGrVZcoyp4fY90iIY7Kv2ws3qZGj3xFBdz8mZ56nZsrofMUPsmqHes7Q+iJ54+fo5om1bLva1p3d
oyi+5Fr12sr4KULf4aa3FiV00hvulkfmS9YI9VgANwkEE1wE7/WEqh5X6jXXoHFyzsVmDpUsopNS
PvJ+8w0nnfWh04aTdu0v01ar4tHbDypQ1JuwP714Ul31bsfoAWrYKPpqu3CfVD1WJi52FnEYp7WQ
B6bJscc7OvoYOaqy6sG7u3jqdZfjkui84w2q9FtTbObgftDnXjOYwEnd64NiVjtgvItg5Ls6+fTo
g9qikur+qkwzh0oW0UkpH3m/+YmdFMZUN02G5TlBLGKqiwZqSSRr2/qf2WpI+DopHAQVWexix95q
Po76EZXsNQOjq4o8s/vchfLXuaUSikp+nTr7H0574VfS8DRVk8JG4adntYtOwGPVrLVPP2Vx0uXf
OdVlvItg5LuCdcLTV/+i2jjIE+vI2OfNHCpZRCelfIRvfiwwoMROigZm6ygeUV02baVqT/2mCG74
p65qdt/kbOnQRU17fZ0UFWjLttHFpdvkjHOVK9VuqLxm+ffRVUUeJ4XtGlZbVPKzU5SH2Eu9dPWc
tPutypT1E9hY3FdMizbLf57kU6iWxUmhzKHqt6VFcS6Czo9FF8iv/6guznEnqEcB8OIbh5h7oZJI
dFLqUGrJt8qw3F+GaqHgivePiLRguI1alvJWaMGXylK9yYZgQy99Yga1Fm9RPx1FfepdZaj/KPWc
1BuP56SJ5XsR4gkFMg6yjMmUtaKTUlZoeoH6TZU3nlioNB+ZZwYPQlkjog9zY3VwTkpVQNFJKSqu
hk+V7FwzSFFe0UkpiqLCik5KURQVVnRSiqKosKKTUhRFhRWdlKIoKqzopBRFUWFFJ6UoigorOilF
UVRYla+TEkJIxcB0wPgESCXksBBoQBNyWOAYJbZDJyX2wzFKbIdOSuyHY5TYDp2U2A/HKLEdOimx
H45RYjt0UmI/HKPEduikxH44Ront0EmJ/XCMEtuhkxL74RgltkMnJfbDMUpsh05K7IdjlNgOnZTY
D8cosR06KbEfjlFiO3RSYj8co8R26KTEfjhGie3QSYn9cIwS26GTEvvhGCXWUVBQkJeXt2/fPr3o
OikiiGNtNJUQO6CTEuuAY6anp2dkZBQWFkZ+c1K0EUHcdVhC7IFOSmwkJycHBlqpUqWsrCw08Ik2
GoibqYRYAJ2U2Mj27dtTUlKkNIggbqYSYgF0UmIpmZmZhpMiYiYRYgd0UmIphYWFhpPqx6aEWAid
lNhLRkaGa6Nom6sJsQY6KbGX/Px810nRNlcTYg10UmIvxcXFqampsFF8om2uJsQa6KTEarKzs+Gk
+DRXEGITdFJiNevXr69cuTI+zRWE2ASdlNjOpEmTzBAhlhHMSd3H/4QQcsRjOmB8AqRGtJMWRSiK
oo580UkpiqLCik5KURQVVnRSiqKosKKTUhRFhRWdlKIoKqzopBRFUWFFJ6UoigorOikV1cJNUrDL
DB5CrflVlm03g0mn8r5KhhZtNiOUhaKTVlxdlSmTXisVqXOyPP2OmXYI1XOw/GWK075pmDQ+S2o3
lBZt5KHnVOSt3dLvAbVYN03OvUSmvmFubonK+yoZwqXIW2kGKdtEJ624Sm8mDz5TKjK9QFb920w7
VFq8RWrWU3apFx+epfzolS9k3Bw5pqryTewa5v7kCpm3Xgb/r6RUkwVfmp3YoN/ZSScvkf8+3wxS
tolOWnHlddL+o+TVjU6773CZ8Z5cN0gu7B5Ne2e/DJ8ql/WULv3k6cLoho/Mk6tvlvad5cYh8to3
0R6ee1f1cHkvtThwrPQYUGp3rs66QIZMNIMNTlVu60125XuEUK+7oveD8S/JhFdj8tfKtbfLn7rK
I/PVDP3W0ap9zwR1Xt7+ocJ96nw73iCXXCsP/M0Juk7qe9bQE8vlihul3dVqX/M/94/Eu5LeTBxD
am2Z81GpA6NsE5204srrpLHVVo1a0rSVjJ2pasbjU2XyUhXE1/68jqpsRPzEOsqYdPKgHHn0FZm2
RnnKKc2lcK/TQ6OWMvpZyV2oFmGXY2aU2t1rW2X2P2RkvtSsL/M3lFq1dJv8x9HKKGODhnyPEKp0
lPzf1077+jvUYwQ3v1lryXlR7bFKinToIkMeV8dWN808MFdX9pEzz5PcReqU7/yrE3Svku9Z40T+
8yTJeUEV+A9MU5fIGymKcyV9MyHcKu561Dw2yirRSSuuDuik+ErrNvwoc6gqkTDjfnOnE4SPwA7c
bTFtRz378mdqCj97ndMDbNRNqFFTGYS7CN32sDoGTO1vvr9UVfj2HmlzqXKx2GSvvEeo2wmcFDaq
2+dcrIo+3R4wRjr3LdWz1tx/ydHHKE834rFXyXvWT61S1fSKH6L53ki8K+nN1LoxW7rdYgYpq0Qn
rbg6oJPO/MBpYxaMb/JjC+ToKmqr9KaS1kRqNVBVlU6AFaKubNVB2l4h1arLlGVOD25VBSGOyX7s
7rSWbVd9urN7VHaYSv/x8gO/H/ceoW4ncFI3H7vIznXaf5miir7YnrVQb+JMvXH3KvmeNSbjnf4s
VY9VDzfvfkydhTcS70p6M/Ue+w4/8H2FOryik1ZcHdBJZ33otOFTXfvLtNWq8vL2g1oM9eaqHc7i
SXXVSxKjB6hhI/OnAq563eU4Bayk4w3S+kJZU2zmeOU9Qt3GzN192tvpplJO6ubDSV3vhpP+qVup
nrUwbccm3ri+SvHOWgvxCa9KozPVo2dvJN6V9GbqCG4Sfe410yirRCetuArqpHC3umkyLM8JYlG/
Enm6UCVr7/uf2erv7uukV2XKoHG/bfurekKq2wu/koanKV+DjcJPz2oXnfa6yl0of51rBr1HqNun
nuE8VVi8RZndQTspzggXxD1f95ew+irFO2vs9PUfncze96ij8kbiXUlvpm43a+1z+pRVopNWXMFJ
Y4GLJXZSNDBbP62Fqi6btlJV2IjpKvjOfvUGHPPcJmer1ziYsfo66cTF0rKt0175kyrKkFAvXT0n
7X6rmtTP31DqeID7mgUufM3AaFdavkdYVPJK/djj1csuHA8s8qCdtKjkJgGX/6/GqkKExeugvkrx
zhqnWb2GCkK4wi9/5hMpinMlfTMXbVavocpSpFOHUXRSKrCWfKtcz/1lqBZKy8T/GgfWA3eL9VaU
YPPWq/rUm2worYm89IkZTKDVv6j3RShyvasOQjgvnJ03XhTnrAt2qfNauCn6Gs0b0fJeSW8m5vh9
h5u7oGwTnZT6/TS9QP1iyRtPLJSrKDO98QqirBHRp7GUtaKTUpSqK9Wb9NLyFpsUFU90UoqiqLCi
k1IURYUVnZSiKCqs6KQURVFhRSelKIoKKzopRVFUWNFJKYqiwopOSlEUFVbl66SEEFIxMB0wPgFS
CSGE+EInJYSQsNBJCSEkLHRSQggJC52UEELCQiclhJCw0EkJISQsdFJCCAkLnZQQQsJCJyWEkLDQ
SQkhJCx0UkIICQudlBBCwkInJYSQsPw/lpixsZ4n73gAAAAASUVORK5CYII=
"></p>

In [20]:
class GNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):            # num_node_features, 64, 2
        super(GNN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.linear1 = Linear(hidden_channels, hidden_channels//2)
        self.linear2 = Linear(hidden_channels//2, out_channels)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = global_mean_pool(x, batch)
        x = self.linear1(x)
        return self.linear2(x)

#### Задача 3 [2 балла]

Реализуйте модель `GINConv` согласно предлагаемой архитектуры, используйте пакет `torch-geometric`

<p style="text-align: center"><img src="...
"></p>

In [21]:
class GIN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):            # num_node_features, 64, 2
        super(GIN, self).__init__()
        self.seq1 = Sequential(Linear(in_channels, hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINConv(self.seq1)

        self.seq2 = Sequential(Linear(hidden_channels, hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINConv(self.seq2)

        self.linear1 = Linear(hidden_channels, hidden_channels//2)
        self.linear2 = Linear(hidden_channels//2, out_channels)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = global_mean_pool(x, batch)
        x = self.linear1(x)
        return self.linear2(x)

Ниже вам предлагаются сервисные функции для тренировки и тестированияя полученной модели

In [22]:
# Training loop
def train():
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.batch)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

# Testing loop
def test(loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.batch)
            pred = out.argmax(dim=1)
            correct += (pred == data.y).sum().item()
    return correct / len(loader.dataset)

Наcтройте модели

In [28]:
model     = GNN(dataset.num_node_features, 64, dataset.num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

In [29]:
gnn_train_acc, gnn_test_acc = [], []

# Main training and testing process
for epoch in range(1, 301):
    loss = train()
    train_acc, test_acc = test(train_loader), test(test_loader )
    gnn_train_acc.append(train_acc)
    gnn_test_acc .append(test_acc )
    if epoch % 50 == 0:
        print(f"Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")

Epoch: 050, Loss: 0.4647, Train Acc: 0.7285, Test Acc: 0.6757
Epoch: 100, Loss: 0.4216, Train Acc: 0.8013, Test Acc: 0.6757
Epoch: 150, Loss: 0.4005, Train Acc: 0.7947, Test Acc: 0.6757
Epoch: 200, Loss: 0.4095, Train Acc: 0.8079, Test Acc: 0.6757
Epoch: 250, Loss: 0.3805, Train Acc: 0.8146, Test Acc: 0.7027
Epoch: 300, Loss: 0.3544, Train Acc: 0.8079, Test Acc: 0.7027


In [46]:
model = GIN(dataset.num_node_features, 64, dataset.num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss()

In [49]:
gin_train_acc, gin_test_acc = [], []

for epoch in range(1, 301):
    loss = train()
    train_acc, test_acc = test(train_loader), test(test_loader )
    gin_train_acc.append(train_acc)
    gin_test_acc .append(test_acc )
    if epoch % 50 == 0:
        print(f"Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")

Epoch: 050, Loss: 0.0813, Train Acc: 0.9139, Test Acc: 0.7568
Epoch: 100, Loss: 0.2436, Train Acc: 0.8344, Test Acc: 0.7297
Epoch: 150, Loss: 0.1285, Train Acc: 0.9073, Test Acc: 0.8378
Epoch: 200, Loss: 0.1012, Train Acc: 0.9139, Test Acc: 0.8919
Epoch: 250, Loss: 0.1454, Train Acc: 0.8808, Test Acc: 0.7027
Epoch: 300, Loss: 0.1334, Train Acc: 0.8940, Test Acc: 0.7297


#### Задача 4 [2 балла]

Выполните анализ полученных реультатов, опишите процессы агрегации информации `GCNConv`, `GINConv`, сравните качество моделей.

In [50]:
import plotly.graph_objects as go
import plotly.express as px

fig = go.Figure()

fig.add_trace(go.Scatter(name="gcn_train_acc", mode="lines", x=list(range(len(gnn_train_acc))), y=gnn_train_acc  ))
fig.add_trace(go.Scatter(name="gcn_test_acc ", mode="lines", x=list(range(len(gnn_test_acc ))), y=gnn_test_acc   ))
fig.add_trace(go.Scatter(name="gin_train_acc", mode="lines", x=list(range(len(gin_train_acc))), y=gin_train_acc  ))
fig.add_trace(go.Scatter(name="gin_test_acc ", mode="lines", x=list(range(len(gin_test_acc ))), y=gin_test_acc   ))

fig.show()

In [ ]:
# GIN справился сильно лучше, чем GCN (max gin_test_acc = 0.891, max gcn_test_acc = 0.756)
# вообще видим, что test_acc у GIN идет на равне с train_acc GCN

In [ ]:
# на самом деле мне кажется надо более тонко отнестись к процессу обучения, но я с этим опыта не имела - очень прыгаюие графики, у GIN вообще тенденция к резким спадам качества(

In [147]:
# "опишите процессы агрегации информации GCNConv, GINConv"

# GCN - суммируем информацию среди всех соседей узла (включая, конечно, сам узел) и аггрегируем их информацию, при этом полученные значения делятся на степени вершин.
# Деление на степень приводит к тому, что мы получаем 'mean-based' aggregation. Это относительно плохо, потому как усреднение приводит к потере чувствительности


# GIN - решает проблему нечувствительности GCN - здесь агрегация = чистая сумма фичей узла и его соседей, для рассматриваемого узла часто используют
# особый trainable параметр (оценка важности информации рассмматриваемого узла среди его соседей). Еще, кажется, здесь идет агрегация информации от разных слоев MLP, котоыре используют для определения фичей графа


# в нашей задаче лучше пользоваться GIN - нам важна чувствительность слоев к отдельным функциональным группам молекул (= подграфам)

####  Задача 5 (Бонус) [3 балла]

Предложите свою модель, которая позволит получить более качественные результаты

In [57]:
# увы, сессия сьела :(